In [ ]:
# 03_A_PyTorch.ipynb Cell 1: build the training stamps (APASS stars cut out of the real plates).
# Needs rcb_starnet.py in the project root next to rcb_paths.py. Safe to re-run: finished stars are skipped.
import os, sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from astropy.wcs import WCS

# Paths come from rcb_paths.py in the project root (found by walking up from this notebook's folder, or set RCB_PROJECT_ROOT)
_here = Path.cwd().resolve()
_cands = ([Path(os.environ['RCB_PROJECT_ROOT'])] if os.environ.get('RCB_PROJECT_ROOT') else []) + [_here, *_here.parents]
_root = next((p for p in _cands if (p / 'rcb_paths.py').exists()), None)
if _root is None:
    raise FileNotFoundError('rcb_paths.py not found. Put it in the project root, or set RCB_PROJECT_ROOT.')
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
import importlib, rcb_paths, rcb_starnet as rs
importlib.reload(rcb_paths); importlib.reload(rs)
BASE_DATA_DIR = rcb_paths.DATA_DIR
CACHE = BASE_DATA_DIR / '_ai_03_A'
CACHE.mkdir(parents=True, exist_ok=True)

RUN_ONLY_THESE_STARS = None    # None = every star that has SISaP results; or e.g. ["UW_Cen"]
PLATES_PER_STAR = 300          # plates sampled per star (published SISaP plates only)
STARS_PER_PLATE = 36           # APASS stars cut out per plate
REBUILD = False                # True = redo stars that already have a stamp file
SEED = 0

def read_target(star_dir):
    for f in (star_dir / '02_A' / 'target_coord.txt', star_dir / 'DASCH' / 'target_coord.txt'):
        if f.exists():
            p = f.read_text().split()
            if len(p) >= 2:
                return float(p[0]), float(p[1])
    return None

star_dirs = sorted([d for d in BASE_DATA_DIR.iterdir() if d.is_dir() and not d.name.startswith('_')
                    and (d / 'DASCH' / 'cutouts').is_dir() and (d / '02_A' / 'photometry_results.csv').exists()
                    and (d / '02_A' / 'apass_field.csv').exists()])
if RUN_ONLY_THESE_STARS is not None:
    star_dirs = [d for d in star_dirs if d.name in RUN_ONLY_THESE_STARS]
print(f'{len(star_dirs)} star folder(s) with cutouts, SISaP results and an APASS field')

summary = []
for si, sd in enumerate(star_dirs):
    out_f = CACHE / f'stamps_{sd.name}.npz'
    if out_f.exists() and not REBUILD:
        z = np.load(out_f, allow_pickle=True)
        summary.append({'star': sd.name, 'plates': len(z['files']), 'stars': int((z['kind'] == 1).sum()), 'artifacts': int((z['kind'] == 2).sum()), 'no star': int((z['kind'] == 3).sum()), 'note': 'already built'})
        continue
    rng = np.random.default_rng(SEED + si)
    pm = pd.read_csv(sd / '02_A' / 'photometry_results.csv')
    pub = pm[pm['tier'].isin(['ok', 'uncertain']) & pm['mag'].notna()].copy()
    if len(pub) > PLATES_PER_STAR:
        pub = pub.sample(PLATES_PER_STAR, random_state=SEED)
    cat = pd.read_csv(sd / '02_A' / 'apass_field.csv')
    iso = rs.isolated_flags(cat['ra'], cat['dec'], cat['Bmag'])
    tgt = read_target(sd)
    bar = rs.Bar(len(pub), sd.name)
    S, B, kind, C, CM, ctx, files = [], [], [], [], [], [], []
    for i, (_, row) in enumerate(pub.iterrows()):
        bar.update(i, f'{sd.name} ({si + 1}/{len(star_dirs)})')
        f = sd / 'DASCH' / 'cutouts' / str(row['file'])
        if not f.exists():
            continue
        try:
            data, hdr = rs.read_plate(f)
            wcs = WCS(hdr)
            if not wcs.has_celestial:
                continue
            plate = rs.Plate(data)
            x, y = wcs.all_world2pix(cat['ra'].values, cat['dec'].values, 0)
            applied = str(row.get('wcs_applied', '')).lower() == 'true'
            dx, dy = (float(row['wcs_dx']), float(row['wcs_dy'])) if applied and np.isfinite(row.get('wcs_dx', np.nan)) else (0.0, 0.0)
            x, y = x + dx, y + dy
            ex = None
            if tgt is not None:
                tx, ty = wcs.all_world2pix(tgt[0], tgt[1], 0)
                ex = (float(tx) + dx, float(ty) + dy)
            lim = row.get('dasch_limit', np.nan)
            tp = rs.training_plate(plate, x, y, cat['Bmag'].values, iso, ex, STARS_PER_PLATE, 4, rng)
            if tp is None:
                continue
            S.append(tp['S']); B.append(tp['B']); kind.append(tp['kind']); C.append(tp['C']); CM.append(tp['CM'])
            ctx.append(plate.ctx(lim)); files.append(str(row['file']))
        except Exception as e:
            print(f'  {sd.name} {row["file"]}: {type(e).__name__}: {e}')
    bar.update(len(pub), 'done')
    if not files:
        print(f'{sd.name}: no usable plates')
        continue
    np.savez(out_f, S=np.stack(S), B=np.stack(B), kind=np.stack(kind), C=np.stack(C), CM=np.stack(CM), ctx=np.stack(ctx), files=np.array(files))
    kk = np.stack(kind)
    summary.append({'star': sd.name, 'plates': len(files), 'stars': int((kk == 1).sum()), 'artifacts': int((kk == 2).sum()), 'no star': int((kk == 3).sum()), 'note': 'built'})
    print(f'{sd.name}: {len(files)} plates, {(kk == 1).sum():,} APASS stamps, {(kk == 2).sum():,} artifact and {(kk == 3).sum():,} empty-sky stamps -> {out_f.name}')

print('\nTRAINING STAMPS')
print(pd.DataFrame(summary).to_string(index=False))

In [ ]:
# 03_A_PyTorch.ipynb Cell 2: train StarNet on the stamps from Cell 1. Saves the model to data/_ai_03_A/starnet_03_A.pt
import os, sys, json, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

_here = Path.cwd().resolve()
_cands = ([Path(os.environ['RCB_PROJECT_ROOT'])] if os.environ.get('RCB_PROJECT_ROOT') else []) + [_here, *_here.parents]
_root = next((p for p in _cands if (p / 'rcb_paths.py').exists()), None)
if _root is None:
    raise FileNotFoundError('rcb_paths.py not found. Put it in the project root, or set RCB_PROJECT_ROOT.')
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
import importlib, rcb_paths, rcb_starnet as rs
importlib.reload(rcb_paths); importlib.reload(rs)
CACHE = rcb_paths.DATA_DIR / '_ai_03_A'

# Stars left out of training, so Cell 3 shows how it does on stars it has never seen. [] = train on everything.
HOLDOUT_STARS = ['S_Aps', 'V_CrA', 'V348_Sgr']
EPOCHS = 20
PLATES_PER_BATCH = 16
LEARNING_RATE = 2e-3
DEBUG_RUN = False              # True = 2 epochs on 200 plates, to check that it runs
SEED = 0

files = sorted(CACHE.glob('stamps_*.npz'))
use = [f for f in files if f.stem.replace('stamps_', '') not in HOLDOUT_STARS]
if not use:
    raise RuntimeError('No stamp files to train on. Run Cell 1 first.')
parts = [np.load(f, allow_pickle=True) for f in use]
data = {k: np.concatenate([p[k] for p in parts]) for k in ('S', 'B', 'kind', 'C', 'CM', 'ctx')}
if DEBUG_RUN:
    keep = np.random.default_rng(SEED).permutation(len(data['S']))[:200]
    data = {k: v[keep] for k, v in data.items()}
    EPOCHS = 2
trained = [f.stem.replace('stamps_', '') for f in use]
print(f'Training on {len(data["S"]):,} plates from {len(trained)} stars: {trained}')
print(f'Held out: {HOLDOUT_STARS}   device: {rs._device()}')
print(f'APASS stamps {(data["kind"] == 1).sum():,}, artifact stamps {(data["kind"] == 2).sum():,}, empty-sky stamps {(data["kind"] == 3).sum():,}')

agent = rs.StarAgent()
agent.meta.update({'trained_stars': trained, 'holdout': HOLDOUT_STARS, 'debug': DEBUG_RUN})
log = agent.fit(data, epochs=EPOCHS, plates_per_batch=PLATES_PER_BATCH, lr=LEARNING_RATE, seed=SEED)
model_path = CACHE / ('starnet_03_A_debug.pt' if DEBUG_RUN else 'starnet_03_A.pt')
agent.save(model_path)
log.to_csv(CACHE / 'train_log.csv', index=False)
print(f'\nSaved {model_path}')

# Held-out plates: scatter of (network - APASS) after each plate's zero point is removed, by APASS magnitude
res, Bv = agent.val_resid
print('\nVALIDATION PLATES (5% of plates, never trained on): scatter of one APASS star, mag')
bins = [(8, 11), (11, 12), (12, 13), (13, 14), (14, 15), (15, 20)]
rows = []
for lo, hi in bins:
    m = (Bv >= lo) & (Bv < hi)
    if m.sum() > 20:
        rows.append({'APASS B': f'{lo}-{hi}', 'stars': int(m.sum()), 'robust sd': round(rs._rsd(res[m] - np.median(res[m])), 3)})
print(pd.DataFrame(rows).to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(log['epoch'], log['train_loss'], label='train'); ax[0].plot(log['epoch'], log['val_loss'], label='validation')
ax[0].set_xlabel('epoch'); ax[0].set_ylabel('loss'); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(Bv, res - np.median(res), '.', ms=2, alpha=0.3)
ax[1].set_ylim(-1.5, 1.5); ax[1].set_xlabel('APASS B'); ax[1].set_ylabel('network - APASS (zero point removed)'); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig(CACHE / 'train_curve.png', dpi=110); plt.show()

In [ ]:
# 03_A_PyTorch.ipynb Cell 3: apply the trained StarNet to every plate of every star, merge with SISaP, compare.
# Writes <Star>/03_A/photometry_ai.csv, lightcurve_ai.csv, lightcurve_improved.csv and ai_vs_sisap_dasch.png. Resumable per plate.
import os, sys, shelve, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.wcs import WCS

_here = Path.cwd().resolve()
_cands = ([Path(os.environ['RCB_PROJECT_ROOT'])] if os.environ.get('RCB_PROJECT_ROOT') else []) + [_here, *_here.parents]
_root = next((p for p in _cands if (p / 'rcb_paths.py').exists()), None)
if _root is None:
    raise FileNotFoundError('rcb_paths.py not found. Put it in the project root, or set RCB_PROJECT_ROOT.')
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
import importlib, rcb_paths, rcb_starnet as rs
importlib.reload(rcb_paths); importlib.reload(rs)
BASE_DATA_DIR = rcb_paths.DATA_DIR
CACHE = BASE_DATA_DIR / '_ai_03_A'

MODEL_FILE = CACHE / 'starnet_03_A.pt'          # use starnet_03_A_debug.pt after a DEBUG_RUN
RUN_ONLY_THESE_STARS = None                     # None = every star with SISaP results
SHOW_PLOTS = True
CFG_OVERRIDES = {}                              # e.g. {'conf_err': 0.25}
PUB = ('ok', 'uncertain')
# DASCH measured a neighbour on the plates where SISaP's pick was far from the target, so those are left out of the DASCH comparison
NEIGHBOUR_PICK_PX = {'V854_Cen': 10.0}

agent = rs.StarAgent.load(MODEL_FILE)
holdout = agent.meta.get('holdout', [])
model_id = str(MODEL_FILE.stat().st_mtime_ns) + str(CFG_OVERRIDES)
print(f'Model {MODEL_FILE.name}: trained on {len(agent.meta.get("trained_stars", []))} stars, held out {holdout}, device {agent.device}')

star_dirs = sorted([d for d in BASE_DATA_DIR.iterdir() if d.is_dir() and not d.name.startswith('_')
                    and (d / 'DASCH' / 'cutouts').is_dir() and (d / '02_A' / 'photometry_results.csv').exists()
                    and (d / '02_A' / 'apass_field.csv').exists() and (d / '02_A' / 'lightcurve.csv').exists()])
if RUN_ONLY_THESE_STARS is not None:
    star_dirs = [d for d in star_dirs if d.name in RUN_ONLY_THESE_STARS]

def read_target(star_dir):
    for f in (star_dir / '02_A' / 'target_coord.txt', star_dir / 'DASCH' / 'target_coord.txt'):
        if f.exists():
            p = f.read_text().split()
            if len(p) >= 2:
                return float(p[0]), float(p[1])
    return None

pooled = []
for si, sd in enumerate(star_dirs):
    tgt = read_target(sd)
    if tgt is None:
        print(f'{sd.name}: no target_coord.txt, skipped')
        continue
    out = sd / '03_A'
    (out / 'cache').mkdir(parents=True, exist_ok=True)
    pm = pd.read_csv(sd / '02_A' / 'photometry_results.csv').drop_duplicates('file').set_index('file', drop=False)
    pm.index.name = None
    cat = pd.read_csv(sd / '02_A' / 'apass_field.csv')
    iso = rs.isolated_flags(cat['ra'], cat['dec'], cat['Bmag'])
    cat_ra, cat_dec, cat_B = cat['ra'].values[iso], cat['dec'].values[iso], cat['Bmag'].values[iso]
    plates = sorted((sd / 'DASCH' / 'cutouts').glob('*.fits'))
    db = shelve.open(str(out / 'cache' / 'ai_plate_shelf'), flag='c', writeback=False)
    bar = rs.Bar(len(plates), sd.name)
    rows = []
    for i, f in enumerate(plates):
        bar.update(i, f'{sd.name} ({si + 1}/{len(star_dirs)})')
        c = db.get(f.name)
        if c is None or c.get('_model') != model_id:
            c = {'file': f.name, 'jd': np.nan, 'tier': 'reject', 'flags': 'error'}
            try:
                data, hdr = rs.read_plate(f)
                wcs = WCS(hdr)
                r = pm.loc[f.name] if f.name in pm.index else None
                if wcs.has_celestial and r is not None:
                    applied = str(r.get('wcs_applied', '')).lower() == 'true'
                    dx, dy = (float(r['wcs_dx']), float(r['wcs_dy'])) if applied and np.isfinite(r.get('wcs_dx', np.nan)) else (0.0, 0.0)
                    x, y = wcs.all_world2pix(cat_ra, cat_dec, 0)
                    tx, ty = wcs.all_world2pix(tgt[0], tgt[1], 0)
                    plate = rs.Plate(data)
                    lim = r.get('dasch_limit', np.nan)
                    m = agent.measure(plate, x + dx, y + dy, cat_B, float(tx) + dx, float(ty) + dy, lim, CFG_OVERRIDES)
                    c = {'file': f.name, 'jd': float(r['date_jd']) if np.isfinite(r['date_jd']) else np.nan, **m}
                else:
                    c['flags'] = 'no WCS or no SISaP row'
            except Exception as e:
                c['flags'] = f'error: {type(e).__name__}: {e}'
            c['_model'] = model_id
            db[f.name] = c
        rows.append({k: v for k, v in c.items() if k != '_model'})
        if (i + 1) % 50 == 0:
            db.sync()
    db.close()
    bar.update(len(plates), 'done')
    ai = pd.DataFrame(rows)

    # Quoted errors rescaled so same-night pairs have a pull of 1 (needs no DASCH)
    ai['mag_err_raw'] = ai['mag_err']
    pubm = ai['tier'].isin(PUB) & ai['mag'].notna() & ai['jd'].notna()
    scale, npair = rs.pull_scale(ai[pubm])
    ai['mag_err'] = ai['mag_err_raw'] * scale
    ai['err_scale'] = scale
    ai.to_csv(out / 'photometry_ai.csv', index=False)
    ai_pub = ai[pubm].copy()
    ai_pub.rename(columns={'mag': 'magnitude', 'mag_err': 'magnitude_error', 'mag_err_raw': 'magnitude_error_raw'})[
        ['jd', 'magnitude', 'magnitude_error', 'magnitude_error_raw', 'tier', 'conf', 'pos_off', 'p_star', 'n_cal', 'cal_rms', 'reach_gap', 'plate_gap', 'file', 'flags']
    ].sort_values('jd').to_csv(out / 'lightcurve_ai.csv', index=False)

    sis = pd.read_csv(sd / '02_A' / 'lightcurve.csv').rename(columns={'magnitude': 'mag', 'magnitude_error': 'mag_err'})[['file', 'jd', 'mag', 'mag_err', 'conf']]
    ai_t = ai_pub[['file', 'jd', 'mag', 'mag_err', 'conf']]
    imp = rs.merge_points(sis, ai_t)
    imp.to_csv(out / 'lightcurve_improved.csv', index=False)

    pmd = pm[(pm['dasch_status'].astype(str) == 'accepted') & pm['dasch_mag'].notna()]
    if sd.name in NEIGHBOUR_PICK_PX:
        pmd = pmd[~(pd.to_numeric(pmd['pick_dist_px'], errors='coerce') > NEIGHBOUR_PICK_PX[sd.name])]
    dasch = pmd[['file', 'date_jd', 'dasch_mag']].rename(columns={'date_jd': 'jd'})

    print(f'\n{sd.name}   (held out of training: {sd.name in holdout})   AI error scale {scale:.2f} from {npair} same-night pairs')
    print(f'  SISaP {len(sis)} points | AI {len(ai_t)} | improved {len(imp)} (both {int((imp.source == "both").sum())}, sisap only {int((imp.source == "sisap").sum())}, ai only {int((imp.source == "ai").sum())})')
    if len(ai_t) and len(sis):
        tabs = {'SISaP': sis, 'AI': ai_t, 'Improved': imp}
        t_all = rs.compare_methods(tabs, dasch)
        print('  all published points, same plates for every method:')
        print(t_all.round(3).to_string(index=False))
        th = {k: v[v['conf'] == 'high'] for k, v in tabs.items()}
        if all(len(v) > 20 for v in th.values()):
            t_hi = rs.compare_methods(th, dasch)
            print('  high confidence only:')
            print(t_hi.round(3).to_string(index=False))
        t_all['star'], t_all['held_out'] = sd.name, sd.name in holdout
        pooled.append(t_all)
    if SHOW_PLOTS and len(imp):
        fig, (a1, a2) = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
        for src, col in (('sisap', '#7f8c8d'), ('ai', '#27ae60'), ('both', '#2980b9')):
            s = imp[imp['source'] == src]
            a1.errorbar(s['jd'], s['mag'], yerr=s['mag_err'], fmt='o', ms=3, color=col, ecolor=col, elinewidth=0.5, alpha=0.8, label=f'{src} ({len(s)})')
        a1.invert_yaxis(); a1.legend(fontsize=8); a1.grid(alpha=0.3); a1.set_ylabel('B magnitude')
        a1.set_title(f'{sd.name}: improved light curve (SISaP + StarNet)')
        a2.plot(dasch['jd'], dasch['dasch_mag'], 'o', ms=3, color='black', alpha=0.8)
        a2.invert_yaxis(); a2.grid(alpha=0.3); a2.set_xlabel('Julian Date'); a2.set_ylabel('DASCH magnitude'); a2.set_title('DASCH (accepted plates)')
        plt.tight_layout(); plt.savefig(out / 'ai_vs_sisap_dasch.png', dpi=110); plt.show()

print('\n' + '=' * 70)
print('POOLED OVER STARS (same plates for every method)')
print('=' * 70)
if pooled:
    P = pd.concat(pooled)
    rows = []
    for held in (False, True):
        sub = P[P['held_out'] == held]
        if not len(sub):
            continue
        for meth in ('SISaP', 'AI', 'Improved', 'DASCH'):
            s = sub[sub['method'] == meth].dropna(subset=['repeat_sd'])
            if not len(s):
                continue
            w = s['pairs'].values
            row = {'stars': 'held out' if held else 'trained on', 'method': meth, 'n_stars': len(s), 'pairs': int(w.sum()),
                   'repeat_sd': float(np.sqrt((w * s['repeat_sd'].values ** 2).sum() / w.sum()))}
            if 'within_0.5' in s and s['within_0.5'].notna().any():
                ww = s['dasch_plates'].fillna(0).values
                row['within_0.5'] = float((s['within_0.5'].fillna(0).values * ww).sum() / max(ww.sum(), 1))
            rows.append(row)
    print(pd.DataFrame(rows).round(3).to_string(index=False))
    P.to_csv(CACHE / 'pooled_comparison.csv', index=False)